# Qwen3.5-0.8B LoRA v4 Training

Run all cells in order on a T4 GPU runtime. The setup cell clones the repo,
installs dependencies, builds corpus v4, and verifies integrity. The training
cell fine-tunes a fresh LoRA adapter and writes checkpoints to Google Drive.

In [ ]:
import subprocess
import sys
from pathlib import Path
from google.colab import drive

PROJECT_DIR = Path("/content/local-slm-lab")
FPY_DIR = Path("/content/fpy")

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

def run(command, cwd=None):
    print("Running:", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

if not (PROJECT_DIR / ".git").is_dir():
    run(["git", "clone", "https://github.com/AbdullahUsman0/SLM-FineTuning-Testing.git", PROJECT_DIR])
else:
    run(["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", "main"])

if not (FPY_DIR / ".git").is_dir():
    run(["git", "clone", "https://github.com/int-abd-5/fpy.git", FPY_DIR])

run(["git", "-C", FPY_DIR, "fetch", "origin", "04d52c015d1e3ecdefe92b87116f209361509b4b"])
run(["git", "-C", FPY_DIR, "checkout", "--detach", "04d52c015d1e3ecdefe92b87116f209361509b4b"])
run([sys.executable, "-m", "pip", "install", "-r", PROJECT_DIR / "training" / "requirements.txt"])
run([sys.executable, "-m", "pip", "install", "-e", FPY_DIR])
run([sys.executable, "scripts/build-corpus-v4.py"], cwd=PROJECT_DIR)
run([sys.executable, "scripts/verify-corpus-v4.py"], cwd=PROJECT_DIR)
run([sys.executable, "-m", "unittest", "tests.test_corpus_v4", "tests.test_training_helpers", "-v"], cwd=PROJECT_DIR)

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/local-slm-lab")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/FYP-model-runs/qwen35-08b-lora-v4")
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable,
    "training/train_lora.py",
    "--train", "corpus-v4/sft/train.jsonl",
    "--validation", "corpus-v4/sft/validation.jsonl",
    "--output", str(DRIVE_OUTPUT),
    "--epochs", "3",
    "--learning-rate", "5e-5",
    "--early-stopping-patience", "1",
    "--max-length", "3072",
    "--allow-truncation",
    "--resume-from-checkpoint", "auto",
]
print("Starting or resuming:", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)

In [ ]:
import json
from pathlib import Path

root = Path("/content/drive/MyDrive/FYP-model-runs/qwen35-08b-lora-v4")
print("Best adapter exists:", (root / "best-adapter" / "adapter_model.safetensors").is_file())
print("Checkpoints:", sorted(path.name for path in root.glob("checkpoint-*")))
print(json.dumps(json.loads((root / "evaluation.json").read_text()), indent=2))